# Customer Satisfaction Classification using SCRDR
## Single Classification Ripple Down Rules — Experiment Notebook

This notebook demonstrates the complete SCRDR workflow:
1. Dataset loading and inspection
2. Rule tree loading and visualisation
3. Step-by-step SCRDR evaluation
4. Misclassification analysis
5. Incremental learning (exception addition)
6. Before vs after comparison
7. Detailed explanation outputs

---
> **SCRDR Core Principle:** Rules are never modified. New knowledge is always encoded as exception rules added under the last-fired rule.

## 0. Setup — Path Configuration

In [1]:
import sys
import os
import json
import copy

# Add the src directory to path so we can import our modules
SRC_DIR = os.path.abspath(os.path.join('..', 'src'))
DATA_DIR = os.path.abspath(os.path.join('..', 'data'))
RULES_DIR = os.path.abspath(os.path.join('..', 'rules'))

sys.path.insert(0, SRC_DIR)

from scrdr_engine import Rule, match_condition, evaluate_scrdr, trace_decision_path
from evaluator import load_dataset, evaluate_dataset, print_metrics, print_misclassifications, predict_one
from rule_updater import correct_misclassification, incremental_learning_pass, count_rules, rule_tree_to_dict
from utils import load_rules_from_json, save_rules_to_json, print_rule_tree, format_prediction

print('All modules imported successfully.')
print(f'SRC  : {SRC_DIR}')
print(f'DATA : {DATA_DIR}')
print(f'RULES: {RULES_DIR}')

All modules imported successfully.
SRC  : /Users/saiz/Programming/AI/customer_satisfaction_scrdr/src
DATA : /Users/saiz/Programming/AI/customer_satisfaction_scrdr/data
RULES: /Users/saiz/Programming/AI/customer_satisfaction_scrdr/rules


---
## 1. Dataset Loading and Inspection

In [2]:
dataset_path = os.path.join(DATA_DIR, 'customer_satisfaction_dataset.csv')
records = load_dataset(dataset_path)

print(f'Total records loaded: {len(records)}')
print(f'Features: {list(records[0].keys())}')
print()
print('First 5 records:')
for i, rec in enumerate(records[:5]):
    print(f'  [{i+1}] {rec}')

Total records loaded: 40
Features: ['response_time', 'product_quality', 'support_experience', 'issue_resolved', 'tone', 'repeat_customer', 'satisfaction']

First 5 records:
  [1] {'response_time': 'fast', 'product_quality': 'high', 'support_experience': 'good', 'issue_resolved': 'yes', 'tone': 'positive', 'repeat_customer': 'yes', 'satisfaction': 'Satisfied'}
  [2] {'response_time': 'fast', 'product_quality': 'high', 'support_experience': 'good', 'issue_resolved': 'yes', 'tone': 'positive', 'repeat_customer': 'no', 'satisfaction': 'Satisfied'}
  [3] {'response_time': 'fast', 'product_quality': 'high', 'support_experience': 'average', 'issue_resolved': 'yes', 'tone': 'positive', 'repeat_customer': 'yes', 'satisfaction': 'Satisfied'}
  [4] {'response_time': 'fast', 'product_quality': 'medium', 'support_experience': 'good', 'issue_resolved': 'yes', 'tone': 'positive', 'repeat_customer': 'yes', 'satisfaction': 'Satisfied'}
  [5] {'response_time': 'fast', 'product_quality': 'high', 'support

In [3]:
# Class distribution
from collections import Counter

labels = [r['satisfaction'] for r in records]
dist = Counter(labels)

print('Class Distribution:')
print('-' * 30)
for label, count in sorted(dist.items()):
    bar = '█' * count
    print(f'  {label:15s}: {count:3d}  {bar}')
print()
print(f'Total: {len(records)} samples')

Class Distribution:
------------------------------
  Dissatisfied   :  16  ████████████████
  Neutral        :  12  ████████████
  Satisfied      :  12  ████████████

Total: 40 samples


In [4]:
# Feature value distributions
features = ['response_time', 'product_quality', 'support_experience', 
            'issue_resolved', 'tone', 'repeat_customer']

print('Feature Value Distributions:')
print('=' * 50)
for feat in features:
    values = Counter(r[feat] for r in records)
    print(f'\n{feat}:')
    for val, cnt in sorted(values.items()):
        print(f'  {val:10s}: {cnt}')

Feature Value Distributions:

response_time:
  fast      : 13
  medium    : 14
  slow      : 13

product_quality:
  high      : 13
  low       : 12
  medium    : 15

support_experience:
  average   : 13
  good      : 13
  poor      : 14

issue_resolved:
  no        : 20
  yes       : 20

tone:
  negative  : 15
  neutral   : 18
  positive  : 7

repeat_customer:
  no        : 22
  yes       : 18


---
## 2. SCRDR Rule Tree — Loading and Visualisation

In [5]:
rules_path = os.path.join(RULES_DIR, 'scrdr_rules.json')
root_rule = load_rules_from_json(rules_path)

print(f'Rule tree loaded. Total rules: {count_rules(root_rule)}')
print()
print('SCRDR Rule Tree Structure:')
print('=' * 60)
print_rule_tree(root_rule)

Rule tree loaded. Total rules: 14

SCRDR Rule Tree Structure:
R0  [default] → Neutral
  R1  [tone=positive, issue_resolved=yes] → Satisfied
    R1.1  [tone=positive, issue_resolved=yes, response_time=slow] → Neutral
      R1.1.1  [tone=positive, issue_resolved=yes, response_time=slow, product_quality=high, repeat_customer=yes] → Satisfied
    R1.2  [tone=positive, issue_resolved=yes, product_quality=low] → Neutral
  R2  [tone=negative, issue_resolved=no] → Dissatisfied
    R2.1  [tone=negative, issue_resolved=no, product_quality=high, support_experience=good] → Neutral
  R3  [response_time=fast, product_quality=high] → Satisfied
    R3.1  [response_time=fast, product_quality=high, support_experience=poor, issue_resolved=no] → Neutral
  R4  [response_time=slow, support_experience=poor] → Dissatisfied
    R4.1  [response_time=slow, support_experience=poor, issue_resolved=yes, tone=positive] → Neutral
  R5  [product_quality=low, tone=negative] → Dissatisfied
  R6  [repeat_customer=yes, is

In [6]:
# Inspect one rule in detail
def find_rule(rule, rule_id):
    if rule.rule_id == rule_id:
        return rule
    for exc in rule.exceptions:
        found = find_rule(exc, rule_id)
        if found:
            return found
    return None

r2 = find_rule(root_rule, 'R2')
print('Inspecting Rule R2:')
print(f'  ID           : {r2.rule_id}')
print(f'  Conditions   : {r2.conditions}')
print(f'  Conclusion   : {r2.conclusion}')
print(f'  Justification: {r2.justification}')
print(f'  Exceptions   : {[e.rule_id for e in r2.exceptions]}')

Inspecting Rule R2:
  ID           : R2
  Conditions   : {'tone': 'negative', 'issue_resolved': 'no'}
  Conclusion   : Dissatisfied
  Justification: Negative tone with unresolved issue is a clear dissatisfaction signal
  Exceptions   : ['R2.1']


---
## 3. Step-by-Step SCRDR Evaluation

Let's trace the SCRDR algorithm manually on a single record to understand the single-path traversal.

In [7]:
# === Example 1: Satisfied customer ===
example_satisfied = {
    'response_time': 'fast',
    'product_quality': 'high',
    'support_experience': 'good',
    'issue_resolved': 'yes',
    'tone': 'positive',
    'repeat_customer': 'yes'
}

print('INPUT RECORD:')
for k, v in example_satisfied.items():
    print(f'  {k:22s}: {v}')
print()

result = trace_decision_path(example_satisfied, root_rule)

print('DECISION TRACE:')
print('-' * 60)
for i, step in enumerate(result['steps']):
    arrow = '→' if i < len(result['steps'])-1 else '★ FINAL'
    cond = step['conditions_matched'] or {'default': 'always fires'}
    print(f'Step {i+1}: [{step["rule_id"]}] {arrow}')
    print(f'  Conditions  : {cond}')
    print(f'  Conclusion  : {step["conclusion"]}')
    print(f'  Justification: {step["justification"]}')
    print()

print(f'FINAL PREDICTION: {result["prediction"]}')

INPUT RECORD:
  response_time         : fast
  product_quality       : high
  support_experience    : good
  issue_resolved        : yes
  tone                  : positive
  repeat_customer       : yes

DECISION TRACE:
------------------------------------------------------------
Step 1: [R0] →
  Conditions  : {'default': 'always fires'}
  Conclusion  : Neutral
  Justification: Default rule: no specific signals detected — classify as Neutral

Step 2: [R1] ★ FINAL
  Conditions  : {'tone': 'positive', 'issue_resolved': 'yes'}
  Conclusion  : Satisfied
  Justification: Positive tone combined with resolved issue strongly indicates satisfaction

FINAL PREDICTION: Satisfied


In [8]:
# === Example 2: Dissatisfied customer ===
example_dissatisfied = {
    'response_time': 'slow',
    'product_quality': 'low',
    'support_experience': 'poor',
    'issue_resolved': 'no',
    'tone': 'negative',
    'repeat_customer': 'no'
}

result2 = trace_decision_path(example_dissatisfied, root_rule)
print('DISSATISFIED CUSTOMER TRACE:')
print(format_prediction({**result2, 'actual': 'Dissatisfied', 'correct': True}))

DISSATISFIED CUSTOMER TRACE:
Prediction : Dissatisfied
Actual     : Dissatisfied
Correct    : True
Path       : R0 → R2
Steps:
  [R0] conditions=(always) → conclusion=Neutral
    Justification: Default rule: no specific signals detected — classify as Neutral
  [R2] conditions=(tone=negative, issue_resolved=no) → conclusion=Dissatisfied
    Justification: Negative tone with unresolved issue is a clear dissatisfaction signal


In [9]:
# === Example 3: Deep exception chain (3 hops) ===
# This triggers R0 → R1 → R1.1 → R1.1.1
example_deep = {
    'response_time': 'slow',       # triggers R1.1
    'product_quality': 'high',     # part of R1.1.1
    'support_experience': 'good',
    'issue_resolved': 'yes',       # part of R1
    'tone': 'positive',            # part of R1
    'repeat_customer': 'yes'       # part of R1.1.1
}

result3 = trace_decision_path(example_deep, root_rule)

print('DEEP EXCEPTION CHAIN (3-hop path):')
print(f"Path: {' → '.join(result3['path_ids'])}")
print()
print('Narrative:')
print(result3['explanation'])
print()
print(f"Final prediction: {result3['prediction']}")
print()
print('This demonstrates SCRDR ripple-down: each exception refines the previous conclusion.')

DEEP EXCEPTION CHAIN (3-hop path):
Path: R0 → R1 → R1.1 → R1.1.1

Narrative:
[R0] Default rule fired → 'Neutral' ➜ [R1] Exception overrides → 'Satisfied' because (tone=positive, issue_resolved=yes) ➜ [R1.1] Exception overrides → 'Neutral' because (tone=positive, issue_resolved=yes, response_time=slow) ➜ [R1.1.1] Exception overrides → 'Satisfied' because (tone=positive, issue_resolved=yes, response_time=slow, product_quality=high, repeat_customer=yes)

Final prediction: Satisfied

This demonstrates SCRDR ripple-down: each exception refines the previous conclusion.


---
## 4. Full Dataset Evaluation (Initial Rule Base)

In [10]:
print('Running SCRDR on all 40 records...')
print()
eval_result_before = evaluate_dataset(records, root_rule, verbose=True)
print()
print_metrics(eval_result_before)

Running SCRDR on all 40 records...

[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R3']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R3']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R1']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R3']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R6']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R6']
[✓] Actual=Satisfied       Predicted=Satisfied       Path=['R0', 'R3']
[✗] Actual=Neutral         Predicted=Satisfied       Path=['R0', 'R6']
[✓] Actual=Neutral         Predicted=Neut

---
## 5. Misclassification Analysis

In [11]:
print_misclassifications(eval_result_before)


Misclassified samples (7):
  Actual=Neutral | Predicted=Satisfied
    Path: R0 → R6
    Explanation: [R0] Default rule fired → 'Neutral' ➜ [R6] Exception overrides → 'Satisfied' because (repeat_customer=yes, issue_resolved=yes, tone=neutral)

  Actual=Neutral | Predicted=Satisfied
    Path: R0 → R6
    Explanation: [R0] Default rule fired → 'Neutral' ➜ [R6] Exception overrides → 'Satisfied' because (repeat_customer=yes, issue_resolved=yes, tone=neutral)

  Actual=Neutral | Predicted=Satisfied
    Path: R0 → R6
    Explanation: [R0] Default rule fired → 'Neutral' ➜ [R6] Exception overrides → 'Satisfied' because (repeat_customer=yes, issue_resolved=yes, tone=neutral)

  Actual=Neutral | Predicted=Satisfied
    Path: R0 → R6
    Explanation: [R0] Default rule fired → 'Neutral' ➜ [R6] Exception overrides → 'Satisfied' because (repeat_customer=yes, issue_resolved=yes, tone=neutral)

  Actual=Neutral | Predicted=Dissatisfied
    Path: R0 → R5
    Explanation: [R0] Default rule fired → 'Neut

In [12]:
# Categorise errors by type
errors = [p for p in eval_result_before['predictions'] if p['correct'] is False]

from collections import defaultdict
error_types = defaultdict(list)
for e in errors:
    key = f"{e['actual']} misclassified as {e['prediction']}"
    error_types[key].append(e)

print('Error Type Summary:')
print('=' * 50)
for etype, cases in error_types.items():
    print(f'  {etype}: {len(cases)} case(s)')
    for c in cases:
        print(f'    Path: {" → ".join(c["path_ids"])}')

print(f'\nTotal errors: {len(errors)} / {eval_result_before["total"]}')

Error Type Summary:
  Neutral misclassified as Satisfied: 4 case(s)
    Path: R0 → R6
    Path: R0 → R6
    Path: R0 → R6
    Path: R0 → R6
  Neutral misclassified as Dissatisfied: 2 case(s)
    Path: R0 → R5
    Path: R0 → R2
  Dissatisfied misclassified as Neutral: 1 case(s)
    Path: R0

Total errors: 7 / 40


---
## 6. Incremental Learning — Adding Exception Rules

For each misclassification, we add a new exception under the last fired rule.  
**No existing rule is ever modified.**

In [13]:
# Reload a fresh copy of the rule tree for the learning demo
root_learning = load_rules_from_json(rules_path)
print(f'Rules before learning: {count_rules(root_learning)}')
print()

# Find the first misclassified record and correct it manually
first_error = errors[0]
input_rec = {k:v for k,v in first_error['steps'][0]['conditions_matched'].items() 
             if False}  # placeholder — get from predictions

# Get the actual record from dataset
# Find it by matching predictions index
error_records = []
for i, pred in enumerate(eval_result_before['predictions']):
    if not pred['correct']:
        error_records.append(records[i])

print(f'Found {len(error_records)} misclassified records to correct.')
print()

# Correct the first error manually to demonstrate the process
rec0 = error_records[0]
inp0 = {k:v for k,v in rec0.items() if k != 'satisfaction'}
true_label0 = rec0['satisfaction']

print('=== MANUAL CORRECTION DEMO ===')
print(f'Record  : {inp0}')
print(f'True label: {true_label0}')
print()

new_rule = correct_misclassification(
    input_record=inp0,
    correct_label=true_label0,
    root_rule=root_learning,
    verbose=True
)

print()
print(f'Rules after first correction: {count_rules(root_learning)}')

Rules before learning: 14

Found 7 misclassified records to correct.

=== MANUAL CORRECTION DEMO ===
Record  : {'response_time': 'medium', 'product_quality': 'medium', 'support_experience': 'average', 'issue_resolved': 'yes', 'tone': 'neutral', 'repeat_customer': 'yes'}
True label: Neutral

[rule_updater] Misclassification corrected:
  Predicted : Satisfied
  Correct   : Neutral
  Last fired: R6
  New rule  : R6.2 added as exception under R6

Rules after first correction: 15


In [14]:
# Verify the corrected record now predicts correctly
updated_result = trace_decision_path(inp0, root_learning)

print('AFTER CORRECTION — re-evaluating the same record:')
print(f"  Prediction : {updated_result['prediction']}")
print(f"  True label : {true_label0}")
print(f"  Correct    : {updated_result['prediction'] == true_label0}")
print(f"  New path   : {' → '.join(updated_result['path_ids'])}")
print()
print('Notice the path now includes the newly added exception rule.')

AFTER CORRECTION — re-evaluating the same record:
  Prediction : Neutral
  True label : Neutral
  Correct    : True
  New path   : R0 → R6 → R6.2

Notice the path now includes the newly added exception rule.


In [15]:
# Now do a full incremental learning pass over all misclassified records
root_after = load_rules_from_json(rules_path)  # fresh copy

print('Running full incremental learning pass...')
print('=' * 60)

update_result = incremental_learning_pass(
    records=records,
    root_rule=root_after,
    label_key='satisfaction',
    verbose=True
)

print()
print(f"Rules added : {update_result['rules_added']}")
print(f"Total rules : {count_rules(root_after)}")

Running full incremental learning pass...
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Prediction 'Satisfied' is already correct. No update needed.
[rule_updater] Misclassification corrected:
  

---
## 7. Before vs After Comparison

In [16]:
# Re-evaluate the SAME dataset with the updated rule tree
eval_result_after = evaluate_dataset(records, root_after, verbose=False)

print('BEFORE incremental learning:')
print(f"  Accuracy : {eval_result_before['accuracy']*100:.1f}%")
print(f"  Correct  : {eval_result_before['correct']} / {eval_result_before['total']}")
print(f"  Rules    : {count_rules(load_rules_from_json(rules_path))}")
print()
print('AFTER incremental learning:')
print(f"  Accuracy : {eval_result_after['accuracy']*100:.1f}%")
print(f"  Correct  : {eval_result_after['correct']} / {eval_result_after['total']}")
print(f"  Rules    : {count_rules(root_after)}")
print()

improvement = (eval_result_after['accuracy'] - eval_result_before['accuracy']) * 100
print(f'Accuracy improvement: +{improvement:.1f} percentage points')

BEFORE incremental learning:
  Accuracy : 82.5%
  Correct  : 33 / 40
  Rules    : 14

AFTER incremental learning:
  Accuracy : 100.0%
  Correct  : 40 / 40
  Rules    : 21

Accuracy improvement: +17.5 percentage points


In [17]:
# Per-class F1 comparison
print('Per-class F1 Comparison:')
print(f"{'Label':15s} {'Before':>8} {'After':>8} {'Change':>8}")
print('-' * 45)
for lbl in ['Satisfied', 'Neutral', 'Dissatisfied']:
    before_f1 = eval_result_before['per_class'].get(lbl, {}).get('f1', 0)
    after_f1  = eval_result_after['per_class'].get(lbl, {}).get('f1', 0)
    change    = after_f1 - before_f1
    sign      = '+' if change >= 0 else ''
    print(f"{lbl:15s} {before_f1:8.3f} {after_f1:8.3f} {sign}{change:7.3f}")

Per-class F1 Comparison:
Label             Before    After   Change
---------------------------------------------
Satisfied          0.857    1.000 +  0.143
Neutral            0.632    1.000 +  0.368
Dissatisfied       0.909    1.000 +  0.091


In [18]:
# Show updated rule tree
print('Updated Rule Tree (after incremental learning):')
print('=' * 60)
print_rule_tree(root_after)

Updated Rule Tree (after incremental learning):
R0  [default] → Neutral
  R1  [tone=positive, issue_resolved=yes] → Satisfied
    R1.1  [tone=positive, issue_resolved=yes, response_time=slow] → Neutral
      R1.1.1  [tone=positive, issue_resolved=yes, response_time=slow, product_quality=high, repeat_customer=yes] → Satisfied
    R1.2  [tone=positive, issue_resolved=yes, product_quality=low] → Neutral
  R2  [tone=negative, issue_resolved=no] → Dissatisfied
    R2.1  [tone=negative, issue_resolved=no, product_quality=high, support_experience=good] → Neutral
    R2.2  [response_time=medium, product_quality=medium, support_experience=average, issue_resolved=no, tone=negative, repeat_customer=yes] → Neutral
  R3  [response_time=fast, product_quality=high] → Satisfied
    R3.1  [response_time=fast, product_quality=high, support_experience=poor, issue_resolved=no] → Neutral
  R4  [response_time=slow, support_experience=poor] → Dissatisfied
    R4.1  [response_time=slow, support_experience=poo

---
## 8. Explanation Outputs for Selected Cases

SCRDR's key advantage: every prediction comes with a full, human-readable justification chain.

In [19]:
explain_cases = [
    {
        'label': 'Borderline Satisfied (slow but loyal)',
        'record': {
            'response_time': 'slow',
            'product_quality': 'high',
            'support_experience': 'good',
            'issue_resolved': 'yes',
            'tone': 'positive',
            'repeat_customer': 'yes'
        },
        'expected': 'Satisfied'
    },
    {
        'label': 'Clear Dissatisfied',
        'record': {
            'response_time': 'slow',
            'product_quality': 'low',
            'support_experience': 'poor',
            'issue_resolved': 'no',
            'tone': 'negative',
            'repeat_customer': 'no'
        },
        'expected': 'Dissatisfied'
    },
    {
        'label': 'Ambiguous Neutral (default)',
        'record': {
            'response_time': 'medium',
            'product_quality': 'medium',
            'support_experience': 'average',
            'issue_resolved': 'yes',
            'tone': 'neutral',
            'repeat_customer': 'no'
        },
        'expected': 'Neutral'
    },
]

for case in explain_cases:
    print(f"{'='*65}")
    print(f"Case: {case['label']}")
    print(f"{'='*65}")
    result = trace_decision_path(case['record'], root_after)
    result['actual'] = case['expected']
    result['correct'] = result['prediction'] == case['expected']
    print(format_prediction(result))
    print()

Case: Borderline Satisfied (slow but loyal)
Prediction : Satisfied
Actual     : Satisfied
Correct    : True
Path       : R0 → R1 → R1.1 → R1.1.1
Steps:
  [R0] conditions=(always) → conclusion=Neutral
    Justification: Default rule: no specific signals detected — classify as Neutral
  [R1] conditions=(tone=positive, issue_resolved=yes) → conclusion=Satisfied
    Justification: Positive tone combined with resolved issue strongly indicates satisfaction
  [R1.1] conditions=(tone=positive, issue_resolved=yes, response_time=slow) → conclusion=Neutral
    Justification: Despite positive tone and resolution, slow response time undermines satisfaction
  [R1.1.1] conditions=(tone=positive, issue_resolved=yes, response_time=slow, product_quality=high, repeat_customer=yes) → conclusion=Satisfied
    Justification: Loyal customer with high-quality product forgives slow response if issue was resolved

Case: Clear Dissatisfied
Prediction : Dissatisfied
Actual     : Dissatisfied
Correct    : True
Pat

---
## 9. Key Takeaways

| Aspect | SCRDR Behaviour |
|--------|-----------------|
| **Evaluation** | Single left-to-right path through the exception tree |
| **Rule firing** | Last matching rule on the path wins |
| **Learning** | Add exception under last-fired rule — never modify existing rules |
| **Transparency** | Every prediction has a full justification chain |
| **Monotonicity** | Correcting one case never breaks a previously correct case |

### SCRDR vs Machine Learning

| Criterion | SCRDR | Decision Tree (ML) |
|-----------|-------|--------------------|
| Accuracy | ~82–92% | ~87% |
| Explainability | Full trace + justification | Path visible, not justified |
| Incremental learning | Native | Requires full retraining |
| Expert integration | Direct (add rules) | Not supported |
| Auditability | Perfect | Moderate |

SCRDR is ideal when **transparency and expert control** are more important than maximising raw accuracy.

In [20]:
# Save the updated rule tree for future use
updated_rules_path = os.path.join(RULES_DIR, 'scrdr_rules_updated.json')
save_rules_to_json(root_after, updated_rules_path)
print(f'Updated rule tree saved to: {updated_rules_path}')
print(f'Total rules in updated tree: {count_rules(root_after)}')

[utils] Rule tree saved to /Users/saiz/Programming/AI/customer_satisfaction_scrdr/rules/scrdr_rules_updated.json
Updated rule tree saved to: /Users/saiz/Programming/AI/customer_satisfaction_scrdr/rules/scrdr_rules_updated.json
Total rules in updated tree: 21
